# GA (só para funcoes de comunicaçao)

In [117]:
#com valores 60 e 120 hard coded

import random
import numpy as np
import math
from deap import base, creator, tools, algorithms
import matplotlib.pyplot as plt  # Importar matplotlib para plotar gráficos
from functools import partial  # Adicionar no início do código
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

NUM_UAVS = 4
lambda_0 = 0.125
P_INTERFERENCE_DBM = 100
P_NOISE_DBM = -100
BANDWIDTH = 20e6
TRANSMIT_POWER_DBM = 20
MINDIST = 20
RESOLUTION = 20
AREA_SIZE = 100  # Tamanho da área em metros

# Definir os tipos de indivíduos e fitness
creator.create("FitnessMax", base.Fitness, weights=(1.0,))  # Maximizar o fitnessAS
creator.create("Individual", list, fitness=creator.FitnessMax)  # Indivíduo é uma listaas

#ATENCAO JAMMER POSITION NO CREATE INDIVIDUAL

def create_individual(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, epoch_total_length=200, previous_best_solution=None, initial_positions=None, jammer_position=None):
    individual = []
    
    # Definir as posições iniciais de acordo com o epoch
    if epoch == 0:
        start_positions = initial_positions
    else:
        start_positions = []
        for uav in range(num_uavs):
            last_timeslot_idx = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            start_positions.append(previous_best_solution[last_timeslot_idx:last_timeslot_idx + 2])
    
    # Gerar posições finais CONTÍNUAS sem restrição de distância mínima
    final_positions = []
    
    for uav in range(num_uavs):
        final_x = random.uniform(60, 120)
        final_y = random.uniform(0, 60)
        final_positions.append((final_x, final_y))
    
    # Calcular todas as posições intermediárias
    for t in range(num_timeslots):
        for uav in range(num_uavs):
            start_x, start_y = start_positions[uav]
            end_x, end_y = final_positions[uav]
            
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            x = start_x + alpha * (end_x - start_x)
            y = start_y + alpha * (end_y - start_y)
            
            individual.extend([x, y])
    
    return creator.Individual(individual)

def print_communication_values_per_timeslot(individual, num_uavs, num_timeslots, jammer_position):
    all_min_capacities = []
    all_interference_matrices = []
    all_fitness_values = []  # ← ADICIONAR para armazenar fitness de cada timeslot

    for t in range(num_timeslots):
        angles = []
        positions = []
        
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        comm_matrix, interference_matrix = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)
        all_interference_matrices.append(interference_matrix)

        # ← USAR A MESMA LÓGICA DO evaluate_individual
        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))
                    except:
                        pass
        
        # Cálculo usando apenas links usados (IGUAL AO evaluate_individual)
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
            all_min_capacities.extend(capacidades_usadas)  # ← Adicionar todas as capacidades usadas
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Calcular fitness do timeslot (IGUAL AO evaluate_individual)
        alpha, beta = 1.0, 1.0
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        all_fitness_values.append(fitness)

    # Calcular fitness médio (IGUAL AO evaluate_individual)
    if len(all_fitness_values) > 0:
        avg_fitness = sum(all_fitness_values) / len(all_fitness_values)
    else:
        avg_fitness = 0.0

    if len(all_min_capacities) > 0:
        avg_min_capacity = sum(all_min_capacities) / len(all_min_capacities)
    else:
        avg_min_capacity = 0.0

    return all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness  # ← ADICIONAR avg_fitness

# Função para calcular a capacidade de comunicação entre todos os UAVs
def calculate_communication_capacity(antenna_angles, alignments, positions, jammer_position):
    communication_capacity = np.zeros((NUM_UAVS, NUM_UAVS))
    interference_matrix = np.full((NUM_UAVS, NUM_UAVS), P_INTERFERENCE_DBM)

    for i in range(NUM_UAVS):
        null_dir_i = antenna_angles[i]

        dx_jam = jammer_position[0] - positions[i][0]
        dy_jam = jammer_position[1] - positions[i][1]
        dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

        G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

        dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
        P_interf_dBm_i = interference_from_jammer(P_INTERFERENCE_DBM, G_jammer_i, lambda_0, dist_jammer_i)

        interference_matrix[i, :] = P_interf_dBm_i

    for i in range(NUM_UAVS):
        for j in range(NUM_UAVS):
            if i != j:
                null_dir_i = antenna_angles[i]
                null_dir_j = antenna_angles[j]

                dx_ij = positions[j][0] - positions[i][0]
                dy_ij = positions[j][1] - positions[i][1]
                dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                dx_ji = positions[i][0] - positions[j][0]
                dy_ji = positions[i][1] - positions[j][1]
                dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                capacity = calcular_capacidade_link(
                    pos1=positions[i],
                    pos2=positions[j],
                    lambda_0=lambda_0,
                    P_tx_dBm=TRANSMIT_POWER_DBM,
                    G_tx_dB=G_tx,
                    G_rx_dB=G_rx,
                    P_interference_dBm=interference_matrix[i, j],
                    P_noise_dBm=P_NOISE_DBM,
                    bandwidth=BANDWIDTH
                )

                communication_capacity[i][j] = capacity

    return communication_capacity, interference_matrix

def evaluate_individual(individual, num_uavs, num_timeslots, jammer_position):
    # Verificar se há pelo menos uma colisão
    
    # Se não há colisões, calcular o fitness normalmente
    alpha, beta = 1.0, 1.0
    total_fitness = 0

    for t in range(num_timeslots):
        positions = []
        angles = []
        for i in range(num_uavs):
            idx = t * num_uavs * 2 + i * 2
            x = individual[idx]
            y = individual[idx + 1]
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            positions.append(np.array([x, y]))
            angles.append(angle)

        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0] * num_uavs, positions, jammer_position)

        # Avaliar grafo
        G = nx.DiGraph()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])

        # Encontrar links usados através dos caminhos mínimos
        links_usados = set()
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            links_usados.add((u, v))
                            links_usados.add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Fitness sem penalização por colisões (já sabemos que não há colisões)
        fitness = (C_media_total ** alpha) * (C_min_total ** beta)
        total_fitness += fitness

    # Calcular a média do fitness total
    average_fitness = total_fitness / num_timeslots if num_timeslots > 0 else 0
    return (average_fitness,)

def mutate_individual(individual, min_x, max_x, min_y, max_y, mutation_rate, num_uavs, num_timeslots, epoch, timeslot_length, epoch_total_length):
    
    for uav in range(num_uavs):
        if random.random() < mutation_rate:
            # Extrair posições iniciais
            start_x = individual[uav * 2]
            start_y = individual[uav * 2 + 1]
            
            # Gerar nova posição final aleatória sem restrição de distância mínima
            new_final_x = random.uniform(60, 120)
            new_final_y = random.uniform(0, 60)
            
            # Atualizar a posição final
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2] = new_final_x
            individual[(num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1] = new_final_y
            
            # Recalcular posições intermediárias
            for t in range(1, num_timeslots):
                alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                x = start_x + alpha * (new_final_x - start_x)
                y = start_y + alpha * (new_final_y - start_y)
                
                idx_x = t * num_uavs * 2 + uav * 2
                idx_y = t * num_uavs * 2 + uav * 2 + 1
                individual[idx_x] = x
                individual[idx_y] = y
    
    return individual,

def custom_crossover(ind1, ind2, num_uavs):
    num_timeslots = len(ind1) // (num_uavs * 2)
    
    # Trocar posições finais entre os indivíduos sem restrição de distância mínima
    for uav in range(num_uavs):
        if random.random() < 0.8:
            # Índices das posições finais (último timeslot)
            final_idx_x = (num_timeslots - 1) * num_uavs * 2 + uav * 2
            final_idx_y = (num_timeslots - 1) * num_uavs * 2 + uav * 2 + 1
            
            # Trocar as posições finais
            ind1[final_idx_x], ind2[final_idx_x] = ind2[final_idx_x], ind1[final_idx_x]
            ind1[final_idx_y], ind2[final_idx_y] = ind2[final_idx_y], ind1[final_idx_y]
            
            # Recalcular posições intermediárias para ambos os indivíduos
            for ind in [ind1, ind2]:
                start_x = ind[uav * 2]
                start_y = ind[uav * 2 + 1]
                end_x = ind[final_idx_x]
                end_y = ind[final_idx_y]
                
                for t in range(1, num_timeslots - 1):
                    alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
                    idx_x = t * num_uavs * 2 + uav * 2
                    idx_y = t * num_uavs * 2 + uav * 2 + 1
                    ind[idx_x] = start_x + alpha * (end_x - start_x)
                    ind[idx_y] = start_y + alpha * (end_y - start_y)
    
    return ind1, ind2

# Configuração do DEAP atualizada
def setup_deap(num_uavs, num_timeslots, timeslot_length, epoch, min_y, max_y, crossover_rate, mutation_rate, initial_positions, previous_best_solution=None, jammer_position=None, epoch_total_length=300):
    toolbox = base.Toolbox()
    
    # Registrar funções
    toolbox.register("individual", create_individual, 
                     num_uavs=num_uavs, 
                     num_timeslots=num_timeslots, 
                     timeslot_length=timeslot_length, 
                     epoch=epoch, 
                     min_y=min_y, 
                     max_y=max_y,
                     epoch_total_length=epoch_total_length,  # ← ADICIONAR
                     previous_best_solution=previous_best_solution,
                     initial_positions=initial_positions,
                     jammer_position=jammer_position)  # Passar a posição do jammer aqui
    
    # O restante do código permanece o mesmo...

    
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate_individual, num_uavs=num_uavs, num_timeslots=num_timeslots, jammer_position=jammer_position)
    
    # Usar partial para fixar o argumento num_uavs na função custom_crossover
    toolbox.register("mate", partial(custom_crossover, num_uavs=num_uavs))
    
    min_x = epoch * num_timeslots * timeslot_length
    max_x = (epoch * num_timeslots + num_timeslots) * timeslot_length
    
    toolbox.register("mutate", mutate_individual, 
                 min_x=min_x, 
                 max_x=max_x, 
                 min_y=min_y, 
                 max_y=max_y, 
                 mutation_rate=mutation_rate,
                 num_uavs=num_uavs,
                 num_timeslots=num_timeslots,
                 epoch=epoch,
                 timeslot_length=timeslot_length,
                 epoch_total_length=epoch_total_length)  # ← ADICIONAR
    
    toolbox.register("select", tools.selTournament, tournsize=3)  # Seleção por torneio
    
    return toolbox

def genetic_algorithm(num_uavs, num_timeslots, timeslot_length, population_size, generations, crossover_rate, mutation_rate, epoch, initial_positions, previous_best_solution=None, jammer_position=None, min_y=0.0, max_y=5.0, epoch_total_length=300):
    # Configurar o DEAP
    toolbox = setup_deap(
        num_uavs=num_uavs,
        num_timeslots=num_timeslots,
        timeslot_length=timeslot_length,
        epoch=epoch,
        min_y=min_y,
        max_y=max_y,
        crossover_rate=crossover_rate,
        mutation_rate=mutation_rate,
        initial_positions=initial_positions,
        previous_best_solution=previous_best_solution,
        jammer_position=jammer_position,
        epoch_total_length=epoch_total_length  # ← ADICIONAR
    )

    
    # Criar população inicial
    population = toolbox.population(n=population_size)
    
    # Avaliar a população inicial
    fitnesses = list(map(toolbox.evaluate, population))
    for ind, fit in zip(population, fitnesses):
        ind.fitness.values = fit
    
    # Configurar estatísticas para impressão
    stats = tools.Statistics(lambda ind: ind.fitness.values)
    stats.register("avg", np.mean)
    stats.register("std", np.std)
    stats.register("min", np.min)
    stats.register("max", np.max)
    
    # Listas para armazenar os dados de cada geração
    gen_list = []
    avg_list = []
    std_list = []
    min_list = []
    max_list = []
    
    # Executar o algoritmo genético com eaSimple
    for gen in range(generations):
        # Avançar uma geração
        algorithms.eaSimple(
            population, 
            toolbox, 
            cxpb=crossover_rate,  # Probabilidade de cruzamento
            mutpb=mutation_rate,  # Probabilidade de mutação
            ngen=1,               # Apenas uma geração por iteração
            stats=stats,          # Estatísticas para impressão
            verbose=False         # Desativar impressão da tabela para cada geração
        )
        
        # Coletar os dados da geração atual
        record = stats.compile(population)
        gen_list.append(gen)
        avg_list.append(record["avg"])
        std_list.append(record["std"])
        min_list.append(record["min"])
        max_list.append(record["max"])
        
        # Escrever os valores no arquivo
        write_fitness_values(epoch, gen, record["avg"], record["max"], record["min"], record["std"])
    
    # Retornar o melhor indivíduo
    best_individual = tools.selBest(population, k=1)[0]
    return best_individual

# Função para gerar posições iniciais dos UAVs
def generate_initial_positions(num_uavs, min_y, max_y, timeslot_length, manual=False, manual_positions=None):
    if manual and manual_positions is not None:
        return manual_positions  # Usa as posições fornecidas

    positions = []
    max_attempts = 1000  # para evitar loops infinitos

    for _ in range(num_uavs):
        attempts = 0
        while True:
            y = random.uniform(min_y, max_y)
            x = random.uniform(0, timeslot_length)  # entre -timeslot_length e 0
            
            candidate = (x, y)
            
            # Verifica se está longe o suficiente das outras posições já geradas
            if all(np.linalg.norm(np.array(candidate) - np.array(pos)) >= MINDIST for pos in positions):
                positions.append(candidate)
                break
            
            attempts += 1
            if attempts >= max_attempts:
                # Se não conseguir, aceita a posição mesmo assim para evitar bloqueio
                positions.append(candidate)
                break
                
    return positions

def save_best_solution_to_file(best_solution, initial_positions, filename="best_solution_epoch.txt"):
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            # Processar o conteúdo existente se necessário
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        # Se o arquivo não existir, não há conteúdo para ler
        print("O arquivo não existe. Criando um novo arquivo.")

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")


def write_fitness_values(epoch, gen, avg, max_val, min_val, std, filename="fitness_values.txt"):
    
    with open(filename, "a") as file:
        if gen == 0:  # Escrever o cabeçalho no início de cada epoch
            file.write(f"=== Epoch {epoch + 1} ===\n")
            file.write("gen\tavg\tmax\tmin\tstd\n")
        file.write(f"{gen}\t{avg:.4f}\t{max_val:.4f}\t{min_val:.4f}\t{std:.4f}\n")

# Função principal atualizada
def simulate_uavs_with_ga(num_epochs, num_timeslots, timeslot_length, num_uavs, population_size=50, generations=50, crossover_rate=0.9, mutation_rate=0.3, manual_initial_positions=None, jammer_position=None, min_y=0.0, max_y=10.0, epoch_total_length=300):
    # Gerar posições iniciais dos UAVs
    initial_positions = generate_initial_positions(
        num_uavs, min_y, max_y, timeslot_length,
        manual=manual_initial_positions is not None,
        manual_positions=manual_initial_positions
    )

    previous_best_solution = None
    
    for epoch in range(num_epochs):
        # Executar o algoritmo genético
        best_solution = genetic_algorithm(
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            timeslot_length=timeslot_length,
            population_size=population_size,
            generations=generations,
            crossover_rate=crossover_rate,
            mutation_rate=mutation_rate,
            epoch=epoch,
            initial_positions=initial_positions,
            previous_best_solution=previous_best_solution,
            jammer_position=jammer_position,  # Passar a posição do jammer aqui
            min_y=min_y,
            max_y=max_y,
            epoch_total_length=epoch_total_length  # ← ADICIONAR
        )

        # O restante do código permanece o mesmo...


        # Salvar a melhor solução em um arquivo de texto
        save_best_solution_to_file(best_solution, initial_positions)
        
        all_min_capacities, all_interference_matrices, avg_min_capacity, avg_fitness = print_communication_values_per_timeslot(best_solution, num_uavs, num_timeslots, jammer_position)

        # Guardar a melhor solução para a próxima epoch


        # # Calcular e imprimir a coerência dos valores
        # best_fitness = best_solution.fitness.values[0]
        # recalculated_fitness = evaluate_individual(best_solution, num_uavs, num_timeslots, jammer_position)[0]
        # # Ler o último valor máximo do fitness_values.txt
        # with open("fitness_values.txt", "r") as file:
        #     lines = file.readlines()
        #     last_max_fitness = float(lines[-1].split("\t")[2])  # Último max na última linha
        # # Comparar os valores
        # print(f"Fitness da melhor solução: {best_fitness:.4f}")
        # print(f"Fitness recalculado: {recalculated_fitness:.4f}")
        # print(f"Último max no arquivo: {last_max_fitness:.4f}")
        # print(f"São iguais? {'✅' if abs(best_fitness - last_max_fitness) < 1 else '❌'}")
        # # Guardar a melhor solução para a próxima epoch
        # previous_best_solution = best_solution



        previous_best_solution = best_solution

        return best_solution, initial_positions, jammer_position, all_min_capacities, all_interference_matrices, avg_min_capacity


# RL 1

In [65]:
import gymnasium as gym
import numpy as np
from gymnasium import spaces
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

class UAVCommunicationEnv(gym.Env):
    def __init__(self, num_uavs=4, num_timeslots=6):
        super().__init__()
        
        # Parâmetros do ambiente
        self.num_uavs = num_uavs
        self.num_timeslots = num_timeslots
        
        # Áreas reais
        self.initial_area_real = (0, 60, 0, 60)      # x_min, x_max, y_min, y_max
        self.target_area_real = (60, 120, 0, 60)
        
        # Parâmetros de comunicação
        self.lambda_0 = 0.125
        self.P_INTERFERENCE_DBM = 100
        self.P_NOISE_DBM = -100
        self.BANDWIDTH = 20e6
        self.TRANSMIT_POWER_DBM = 20
        
        # Espaço de observação normalizado [0,1]: [uav1_x, uav1_y, ..., uav4_x, uav4_y, jammer_x, jammer_y]
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(10,),  # 4 UAVs * 2 coords + jammer * 2 coords
            dtype=np.float32
        )
        
        # Espaço de ação normalizado [0,1]: [uav1_x_final, uav1_y_final, ..., uav4_x_final, uav4_y_final]
        self.action_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(8,),  # 4 UAVs * 2 coords finais
            dtype=np.float32
        )

    def _normalize_uav_position(self, pos, area_type='initial'):
        """Normaliza posição de UAV para [0,1]"""
        if area_type == 'initial':
            x_min, x_max, y_min, y_max = self.initial_area_real
        elif area_type == 'target':
            x_min, x_max, y_min, y_max = self.target_area_real
        
        norm_x = (pos[0] - x_min) / (x_max - x_min)
        norm_y = (pos[1] - y_min) / (y_max - y_min)
        return np.array([norm_x, norm_y])
    
    def _normalize_jammer_position(self, pos):
        """Normaliza posição do jammer para [0,1]"""
        # Jammer: x ∈ [0,120], y = 500 (fixo)
        norm_x = pos[0] / 120.0  # x entre 0 e 120
        norm_y = 1.0  # y sempre 500, normalizado como 1.0
        return np.array([norm_x, norm_y])
    
    def _denormalize_uav_position(self, norm_pos, area_type='target'):
        """Desnormaliza posição de UAV de [0,1] para coordenadas reais"""
        if area_type == 'initial':
            x_min, x_max, y_min, y_max = self.initial_area_real
        elif area_type == 'target':
            x_min, x_max, y_min, y_max = self.target_area_real
        
        real_x = norm_pos[0] * (x_max - x_min) + x_min
        real_y = norm_pos[1] * (y_max - y_min) + y_min
        return np.array([real_x, real_y])

    def _generate_initial_positions(self):
        """Gera posições iniciais aleatórias"""
        positions = []
        
        for _ in range(self.num_uavs):
            x = np.random.uniform(0, 60)
            y = np.random.uniform(0, 60)
            positions.append(np.array([x, y]))
        
        return np.array(positions)

    def _calculate_reward(self, final_positions_real):
        """Calcula reward usando a mesma lógica do GA, com penalização por colisões."""
        alpha, beta = 1.0, 1.0
        total_fitness = 0

        # Avaliar em cada timeslot (pontos ao longo do caminho)
        for t in range(self.num_timeslots):
            positions = []
            angles = []
            
            # Calcular posições interpoladas para este timeslot
            for i in range(self.num_uavs):
                start_pos = self.uav_positions_real[i]
                end_pos = final_positions_real[i]
                
                # Interpolação linear
                alpha_interp = t / (self.num_timeslots - 1) if self.num_timeslots > 1 else 0
                current_pos = start_pos + alpha_interp * (end_pos - start_pos)
                positions.append(current_pos)
                
                # Calcular ângulo da antena (direção oposta ao jammer)
                angle = np.degrees(np.arctan2(
                    self.jammer_position_real[1] - current_pos[1],
                    self.jammer_position_real[0] - current_pos[0]
                ))
                angles.append(angle)
            
            # Calcular matriz de comunicação
            comm_matrix, _ = self._calculate_communication_capacity(
                angles, [1.0] * self.num_uavs, positions, self.jammer_position_real
            )
            
            # Avaliar grafo (mesma lógica do GA)
            G = nx.DiGraph()
            for i in range(self.num_uavs):
                for j in range(self.num_uavs):
                    if i != j and comm_matrix[i][j] > 0:
                        G.add_edge(i, j, capacity=comm_matrix[i][j])
            
            # Encontrar links usados
            links_usados = set()
            for i in range(self.num_uavs):
                for j in range(self.num_uavs):
                    if i != j:
                        try:
                            path = nx.shortest_path(G, source=i, target=j, 
                                                    weight=lambda u, v, d: 1/d['capacity'])
                            for u, v in zip(path[:-1], path[1:]):
                                links_usados.add((u, v))
                                links_usados.add((v, u))
                        except:
                            pass
            
            # Calcular fitness do timeslot
            capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados 
                                if u < self.num_uavs and v < self.num_uavs]
            
            if capacidades_usadas:
                C_media_total = np.mean(capacidades_usadas)
                C_min_total = min(capacidades_usadas)
            else:
                C_media_total = 0
                C_min_total = 0
            
            fitness = (C_media_total ** alpha) * (C_min_total ** beta)
            total_fitness += fitness

        # # Verificar se há colisões (distâncias menores que 20 entre UAVs)
        # for i in range(self.num_uavs):
        #     for j in range(i + 1, self.num_uavs):
        #         distance = np.linalg.norm(np.array(final_positions_real[i]) - np.array(final_positions_real[j]))
        #         if distance < 20:
        #             return 0  # Retornar 0 se houver colisão

        # Retornar fitness médio
        # Retornar fitness médio
        avg_fit = total_fitness / self.num_timeslots if self.num_timeslots > 0 else 0

        # ✅ Nova forma de reward: compressão logarítmica suave
        reward = np.log10(1.0 + avg_fit)

        return float(reward)

    def _calculate_communication_capacity(self, antenna_angles, alignments, positions, jammer_position):
        """Mesma função do teu código"""
        communication_capacity = np.zeros((self.num_uavs, self.num_uavs))
        interference_matrix = np.full((self.num_uavs, self.num_uavs), self.P_INTERFERENCE_DBM)

        for i in range(self.num_uavs):
            null_dir_i = antenna_angles[i]

            dx_jam = jammer_position[0] - positions[i][0]
            dy_jam = jammer_position[1] - positions[i][1]
            dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

            G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

            dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
            P_interf_dBm_i = interference_from_jammer(self.P_INTERFERENCE_DBM, G_jammer_i, 
                                                    self.lambda_0, dist_jammer_i)

            interference_matrix[i, :] = P_interf_dBm_i

        for i in range(self.num_uavs):
            for j in range(self.num_uavs):
                if i != j:
                    null_dir_i = antenna_angles[i]
                    null_dir_j = antenna_angles[j]

                    dx_ij = positions[j][0] - positions[i][0]
                    dy_ij = positions[j][1] - positions[i][1]
                    dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                    dx_ji = positions[i][0] - positions[j][0]
                    dy_ji = positions[i][1] - positions[j][1]
                    dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                    G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                    G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                    capacity = calcular_capacidade_link(
                        pos1=positions[i],
                        pos2=positions[j],
                        lambda_0=self.lambda_0,
                        P_tx_dBm=self.TRANSMIT_POWER_DBM,
                        G_tx_dB=G_tx,
                        G_rx_dB=G_rx,
                        P_interference_dBm=interference_matrix[i, j],
                        P_noise_dBm=self.P_NOISE_DBM,
                        bandwidth=self.BANDWIDTH
                    )

                    communication_capacity[i][j] = capacity

        return communication_capacity, interference_matrix
    
    def reset(self, initial_positions=None, jammer_position=None, seed=None, options=None):
        super().reset(seed=seed)
        
        # Usar posições fornecidas se existirem
        # if initial_positions is not None:
        #     self.uav_positions_real = np.array(initial_positions)
        # else:
        #     self.uav_positions_real = self._generate_initial_positions()

        # Fixar UAVs em posições conhecidas (por exemplo, formação linear)
        self.uav_positions_real = np.array([
            [10, 10],
            [20, 20],
            [30, 30],
            [40, 40]
        ])
        
        # if jammer_position is not None:
        #     self.jammer_position_real = np.array(jammer_position)
        # else:
        #     self.jammer_position_real = np.array([
        #         np.random.uniform(0, 120),
        #         500.0
        #     ])

        # Fixar jammer numa posição constante
        self.jammer_position_real = np.array([60.0, 500.0])
        
        # Normalizar para observação
        uav_positions_norm = []
        for pos in self.uav_positions_real:
            norm_pos = self._normalize_uav_position(pos, 'initial')
            uav_positions_norm.extend(norm_pos)
        
        jammer_norm = self._normalize_jammer_position(self.jammer_position_real)
        
        obs = np.concatenate([uav_positions_norm, jammer_norm])
        
        return obs.astype(np.float32), {}

    def step(self, action):
        # Desnormalizar ações para coordenadas reais
        action_reshaped = action.reshape(4, 2)
        final_positions_real = []

        for norm_pos in action_reshaped:
            real_pos = self._denormalize_uav_position(norm_pos, 'target')
            final_positions_real.append(real_pos)

        final_positions_real = np.array(final_positions_real)

        # ✅ Guardar cópia das posições iniciais antes de atualizar
        initial_positions_copy = self.uav_positions_real.copy()

        # Calcular reward
        reward = self._calculate_reward(final_positions_real)

        # ✅ Atualizar o estado interno (para refletir as novas posições)
        self.uav_positions_real = final_positions_real

        # ✅ Normalizar o reward (mais informativo)
        #reward = reward / (1e8 + reward)

        # ✅ Nova observação = posições finais (não as iniciais)
        uav_positions_norm = []
        for pos in final_positions_real:
            norm_pos = self._normalize_uav_position(pos, 'target')
            uav_positions_norm.extend(norm_pos)

        jammer_norm = self._normalize_jammer_position(self.jammer_position_real)
        obs = np.concatenate([uav_positions_norm, jammer_norm])

        # Episódio termina (decisão única)
        terminated = True
        truncated = False

        info = {
            'initial_positions_real': initial_positions_copy,  # 👈 corrigido
            'final_positions_real': final_positions_real,
            'jammer_position_real': self.jammer_position_real
        }

        return obs.astype(np.float32), reward, terminated, truncated, info


# Função para salvar (adiciona antes do if __name__ == "__main__":)
def save_best_solution_to_file(initial_positions, final_positions, num_timeslots=6, filename="best_solution_rl.txt"):
    """
    Salva a melhor solução do RL no mesmo formato do GA
    """
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        print("O arquivo não existe. Criando um novo arquivo.")

    # Criar a solução completa (mesmo formato do GA)
    best_solution = []
    
    # Para cada timeslot, calcular posições interpoladas
    for t in range(num_timeslots):
        for uav in range(len(initial_positions)):
            start_pos = initial_positions[uav]
            end_pos = final_positions[uav]
            
            # Interpolação linear (mesma lógica do GA)
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            x = start_pos[0] + alpha * (end_pos[0] - start_pos[0])
            y = start_pos[1] + alpha * (end_pos[1] - start_pos[1])
            
            best_solution.extend([x, y])

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")
    
    print(f"Solução salva com {len(best_solution)} valores")
    return best_solution

# Exemplo de uso
if __name__ == "__main__":
    env = UAVCommunicationEnv()
    
    # Resetar o ambiente (usa posições fixas internas)
    obs, _ = env.reset()
    
    print(f"Observação inicial: {obs}")
    print(f"Shape observação: {obs.shape}")
    
    # Ação aleatória
    action = env.action_space.sample()
    print(f"Ação: {action}")
    
    # Executar a ação
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"\nReward: {reward:.4f}")
    print("\nPosições iniciais reais:")
    print(np.round(info['initial_positions_real'], 2))
    
    print("\nPosições finais reais:")
    print(np.round(info['final_positions_real'], 2))
    
    print(f"\nPosição jammer: {info['jammer_position_real']}")


Observação inicial: [0.16666667 0.16666667 0.33333334 0.33333334 0.5        0.5
 0.6666667  0.6666667  0.5        1.        ]
Shape observação: (10,)
Ação: [0.04273459 0.2308132  0.3584163  0.83409363 0.23786378 0.47399315
 0.32141912 0.5462148 ]

Reward: 6.1689

Posições iniciais reais:
[[10 10]
 [20 20]
 [30 30]
 [40 40]]

Posições finais reais:
[[62.56 13.85]
 [81.5  50.05]
 [74.27 28.44]
 [79.29 32.77]]

Posição jammer: [ 60. 500.]


# treinar com dataset

#Jammer_X,Jammer_Y,Fitness,Média_C_media_total,Média_C_min_total,Initial_X1,Initial_Y1,Initial_X2,Initial_Y2,Initial_X3,Initial_Y3,Initial_X4,Initial_Y4,Final_X1,Final_Y1,Final_X2,Final_Y2,Final_X3,Final_Y3,Final_X4,Final_Y4

In [13]:
import numpy as np
import pandas as pd

csv_path = "dataset_combined_com_permutacoes.csv"
df = pd.read_csv(csv_path, comment='#')  # ignora header/comment

# Colunas esperadas (ex.: Jammer_X,Jammer_Y,Fitness,...,Final_X4,Final_Y4)
# Ajusta nomes se necessário

# Normalization helpers
def norm_initial(pos):
    # pos: (x,y) in [0,60] each
    return [(pos[0] - 0) / 60.0, (pos[1] - 0) / 60.0]

def norm_target(pos):
    # pos: (x,y) in [60,120] x [0,60]
    return [(pos[0] - 60.0) / 60.0, (pos[1] - 0) / 60.0]

def norm_jammer_x(x):
    return x / 120.0

# Construir arrays
N = len(df)
obs = np.zeros((N, 10), dtype=np.float32)
acts = np.zeros((N, 8), dtype=np.float32)

for i, row in df.iterrows():
    # jammer
    jx = row['Jammer_X']
    jy = row['Jammer_Y']  # provavelmente sempre 500
    # normalize jammer
    obs[i, 8] = norm_jammer_x(jx)
    obs[i, 9] = 1.0  # jy fixed -> 1.0 (mantém compatibilidade)

    # initial UAVs (columns Initial_X1, Initial_Y1, ...)
    for u in range(4):
        ix = row[f'Initial_X{u+1}']
        iy = row[f'Initial_Y{u+1}']
        obs[i, 2*u:2*u+2] = norm_initial((ix, iy))

    # final UAVs (target) -> actions (normalize target domain)
    for u in range(4):
        fx = row[f'Final_X{u+1}']
        fy = row[f'Final_Y{u+1}']
        acts[i, 2*u:2*u+2] = norm_target((fx, fy))

# Save for quick loading
np.savez_compressed("dataset_combined_com_permutacoes.npz", obs=obs, acts=acts)
print("Saved:", obs.shape, acts.shape)


Saved: (2400, 10) (2400, 8)


In [67]:
import torch
from torch.utils.data import TensorDataset, DataLoader

data = np.load("uav_dataset_reference_3_com_permutacoes_normalized.npz")

obs = torch.tensor(data["obs"], dtype=torch.float32)
acts = torch.tensor(data["acts"], dtype=torch.float32)
dataset = TensorDataset(obs, acts)
loader = DataLoader(dataset, batch_size=256, shuffle=True)


In [68]:
import torch.nn as nn

class PolicyNetwork(nn.Module):
    def __init__(self, input_dim=10, output_dim=8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim),
            nn.Sigmoid()  # como as ações estão normalizadas em [0,1]
        )

    def forward(self, x):
        return self.net(x)


In [69]:
model = PolicyNetwork()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()  # erro entre ação prevista e ação real do GA


In [9]:
n_epochs = 10
for epoch in range(n_epochs):
    total_loss = 0
    for batch_obs, batch_acts in loader:
        pred_acts = model(batch_obs)
        loss = criterion(pred_acts, batch_acts)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Época {epoch+1}/{n_epochs} - Loss: {total_loss/len(loader):.6f}")


Época 1/10 - Loss: 0.125659
Época 2/10 - Loss: 0.125538
Época 3/10 - Loss: 0.125462
Época 4/10 - Loss: 0.125415
Época 5/10 - Loss: 0.125340
Época 6/10 - Loss: 0.125270
Época 7/10 - Loss: 0.125207
Época 8/10 - Loss: 0.125127
Época 9/10 - Loss: 0.125068
Época 10/10 - Loss: 0.125005


In [11]:
torch.save(model.state_dict(), "policy_pretrained_uav.pth")
print("Modelo pré-treinado salvo com sucesso!")


Modelo pré-treinado salvo com sucesso!


In [70]:
import torch
import numpy as np

# ===== 1. Recriar a mesma arquitetura usada no treino =====
class PolicyNetwork(torch.nn.Module):
    def __init__(self, input_dim=10, output_dim=8):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, 128),
            torch.nn.ReLU(),
            torch.nn.Linear(128, output_dim),
            torch.nn.Sigmoid()  # ações normalizadas [0,1]
        )

    def forward(self, x):
        return self.net(x)

# ===== 2. Carregar modelo salvo =====
model = PolicyNetwork()
model.load_state_dict(torch.load("policy_pretrained_uav.pth", map_location="cpu"))
model.eval()  # modo teste
print("✅ Modelo carregado com sucesso!")

# ===== 3. Carregar uma amostra do dataset =====
data = np.load("dataset_combined_com_permutacoes.npz")
obs = torch.tensor(data["obs"], dtype=torch.float32)
acts_true = torch.tensor(data["acts"], dtype=torch.float32)

# Escolher uma amostra aleatória
idx = np.random.randint(len(obs))
obs_sample = obs[idx].unsqueeze(0)
true_act = acts_true[idx]

# ===== 4. Fazer previsão =====
with torch.no_grad():
    pred_act = model(obs_sample).squeeze(0)

# ===== 5. Mostrar resultados =====
print("\n--- Teste de Policy Pré-Treinada ---")
print(f"Observação (normalizada): {obs_sample.numpy().round(3)}")
print(f"\nAção prevista pela policy: {pred_act.numpy().round(3)}")
print(f"Ação real do GA:           {true_act.numpy().round(3)}")

# (Opcional) diferença média
error = torch.mean((pred_act - true_act).abs()).item()
print(f"\nErro médio absoluto: {error:.5f}")


✅ Modelo carregado com sucesso!

--- Teste de Policy Pré-Treinada ---
Observação (normalizada): [[0.839 0.101 0.022 0.681 0.623 0.66  0.437 0.283 0.556 1.   ]]

Ação prevista pela policy: [0.567 0.5   0.605 0.465 0.577 0.465 0.588 0.497]
Ação real do GA:           [0.077 0.655 0.61  0.268 0.026 0.272 0.299 0.54 ]

Erro médio absoluto: 0.24044


# ambiente com dataset - treino com pesos caregados

In [74]:

# Criar ambiente
env = UAVCommunicationEnv()
obs, _ = env.reset()

# Converter observação para tensor
obs_tensor = torch.tensor(obs, dtype=torch.float32).unsqueeze(0)

# Gerar ação com o modelo pré-treinado
with torch.no_grad():
    pred_action = model(obs_tensor).squeeze(0).numpy()

# Passar essa ação ao ambiente
obs_next, reward, terminated, truncated, info = env.step(pred_action)

print("\n--- Teste no ambiente ---")
print(f"Ação prevista: {np.round(pred_action, 3)}")
print(f"Reward obtido: {reward:.4f}")
print(f"Posições finais reais:\n{np.round(info['final_positions_real'], 2)}")



--- Teste no ambiente ---
Ação prevista: [0.607 0.486 0.607 0.472 0.595 0.468 0.595 0.457]
Reward obtido: 7.6553
Posições finais reais:
[[96.4  29.17]
 [96.45 28.33]
 [95.69 28.1 ]
 [95.7  27.43]]


In [75]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

# Criar o ambiente
env = make_vec_env(UAVCommunicationEnv, n_envs=4)


In [77]:
import torch

model = PPO("MlpPolicy", env, verbose=1)
pretrained_weights = torch.load("policy_pretrained_uav.pth", map_location="cpu")
model.policy.load_state_dict(pretrained_weights, strict=False)
print("✅ Pesos do pré-treino carregados na policy PPO!")

Using cpu device
✅ Pesos do pré-treino carregados na policy PPO!


In [78]:
model.learn(total_timesteps=300_000)
model.save("policy_finetuned.zip")
print("✅ Fine-tuning RL concluído!")

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | 7.69     |
| time/              |          |
|    fps             | 143      |
|    iterations      | 1        |
|    time_elapsed    | 57       |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1           |
|    ep_rew_mean          | 7.59        |
| time/                   |             |
|    fps                  | 86          |
|    iterations           | 2           |
|    time_elapsed         | 188         |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.040747177 |
|    clip_fraction        | 0.467       |
|    clip_range           | 0.2         |
|    entropy_loss         | -11.4       |
|    explained_variance   | 0           |
|    learning_rate        | 0.

# testar ambiente

In [80]:
from stable_baselines3 import PPO
model = PPO.load("policy_finetuned.zip")

In [81]:
env_single = UAVCommunicationEnv()
obs, _ = env_single.reset()


In [98]:
action, _ = model.predict(obs, deterministic=True)
obs, reward, terminated, truncated, info = env_single.step(action)

print("Ação sugerida:", action)
print("Reward obtida:", reward)
print("Posições finais:", info['final_positions_real'])
print("Posição jammer:", info['jammer_position_real'])


Ação sugerida: [0. 0. 0. 0. 0. 0. 0. 0.]
Reward obtida: 16.212070103243125
Posições finais: [[60.  0.]
 [60.  0.]
 [60.  0.]
 [60.  0.]]
Posição jammer: [ 60. 500.]


# novo

In [111]:

import gymnasium as gym
import numpy as np
from gymnasium import spaces
import networkx as nx
from link_capacity import calcular_capacidade_link, interference_from_jammer
from antenna_null import realistic_antenna_gain

class UAVCommunicationEnv(gym.Env):
    def __init__(self, num_uavs=4, num_timeslots=6):
        super().__init__()
        
        # Parâmetros do ambiente
        self.num_uavs = num_uavs
        self.num_timeslots = num_timeslots
        
        # Áreas reais
        self.initial_area_real = (0, 60, 0, 60)      # x_min, x_max, y_min, y_max
        self.target_area_real = (60, 120, 0, 60)
        
        # Parâmetros de comunicação
        self.lambda_0 = 0.125
        self.P_INTERFERENCE_DBM = 100
        self.P_NOISE_DBM = -100
        self.BANDWIDTH = 20e6
        self.TRANSMIT_POWER_DBM = 20
        
        # Espaço de observação normalizado [0,1]: [uav1_x, uav1_y, ..., uav4_x, uav4_y, jammer_x, jammer_y]
        self.observation_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(10,),  # 4 UAVs * 2 coords + jammer * 2 coords
            dtype=np.float32
        )
        
        # Espaço de ação normalizado [0,1]: [uav1_x_final, uav1_y_final, ..., uav4_x_final, uav4_y_final]
        self.action_space = spaces.Box(
            low=0.0,
            high=1.0,
            shape=(8,),  # 4 UAVs * 2 coords finais
            dtype=np.float32
        )

    def _normalize_uav_position(self, pos, area_type='initial'):
        """Normaliza posição de UAV para [0,1]"""
        if area_type == 'initial':
            x_min, x_max, y_min, y_max = self.initial_area_real
        elif area_type == 'target':
            x_min, x_max, y_min, y_max = self.target_area_real
        
        norm_x = (pos[0] - x_min) / (x_max - x_min)
        norm_y = (pos[1] - y_min) / (y_max - y_min)
        return np.array([norm_x, norm_y])
    
    def _normalize_jammer_position(self, pos):
        """Normaliza posição do jammer para [0,1]"""
        # Jammer: x ∈ [0,120], y = 500 (fixo)
        norm_x = pos[0] / 120.0  # x entre 0 e 120
        norm_y = 1.0  # y sempre 500, normalizado como 1.0
        return np.array([norm_x, norm_y])
    
    def _denormalize_uav_position(self, norm_pos, area_type='target'):
        """Desnormaliza posição de UAV de [0,1] para coordenadas reais"""
        if area_type == 'initial':
            x_min, x_max, y_min, y_max = self.initial_area_real
        elif area_type == 'target':
            x_min, x_max, y_min, y_max = self.target_area_real
        
        real_x = norm_pos[0] * (x_max - x_min) + x_min
        real_y = norm_pos[1] * (y_max - y_min) + y_min
        return np.array([real_x, real_y])

    def _generate_initial_positions(self):
        """Gera posições iniciais aleatórias"""
        positions = []
        
        for _ in range(self.num_uavs):
            x = np.random.uniform(0, 60)
            y = np.random.uniform(0, 60)
            positions.append(np.array([x, y]))
        
        return np.array(positions)

    def _calculate_reward(self, final_positions_real):
        """Calcula reward usando a mesma lógica do GA, com penalização por colisões."""
        alpha, beta = 1.0, 1.0
        total_fitness = 0

        # Avaliar em cada timeslot (pontos ao longo do caminho)
        for t in range(self.num_timeslots):
            positions = []
            angles = []
            
            # Calcular posições interpoladas para este timeslot
            for i in range(self.num_uavs):
                start_pos = self.uav_positions_real[i]
                end_pos = final_positions_real[i]
                
                # Interpolação linear
                alpha_interp = t / (self.num_timeslots - 1) if self.num_timeslots > 1 else 0
                current_pos = start_pos + alpha_interp * (end_pos - start_pos)
                positions.append(current_pos)
                
                # Calcular ângulo da antena (direção oposta ao jammer)
                angle = np.degrees(np.arctan2(
                    self.jammer_position_real[1] - current_pos[1],
                    self.jammer_position_real[0] - current_pos[0]
                ))
                angles.append(angle)
            
            # Calcular matriz de comunicação
            comm_matrix, _ = self._calculate_communication_capacity(
                angles, [1.0] * self.num_uavs, positions, self.jammer_position_real
            )
            
            # Avaliar grafo (mesma lógica do GA)
            G = nx.DiGraph()
            for i in range(self.num_uavs):
                for j in range(self.num_uavs):
                    if i != j and comm_matrix[i][j] > 0:
                        G.add_edge(i, j, capacity=comm_matrix[i][j])
            
            # Encontrar links usados
            links_usados = set()
            for i in range(self.num_uavs):
                for j in range(self.num_uavs):
                    if i != j:
                        try:
                            path = nx.shortest_path(G, source=i, target=j, 
                                                    weight=lambda u, v, d: 1/d['capacity'])
                            for u, v in zip(path[:-1], path[1:]):
                                links_usados.add((u, v))
                                links_usados.add((v, u))
                        except:
                            pass
            
            # Calcular fitness do timeslot
            capacidades_usadas = [comm_matrix[u][v] for u, v in links_usados 
                                if u < self.num_uavs and v < self.num_uavs]
            
            if capacidades_usadas:
                C_media_total = np.mean(capacidades_usadas)
                C_min_total = min(capacidades_usadas)
            else:
                C_media_total = 0
                C_min_total = 0
            
            fitness = (C_media_total ** alpha) * (C_min_total ** beta)
            total_fitness += fitness

        # # Verificar se há colisões (distâncias menores que 20 entre UAVs)
        # for i in range(self.num_uavs):
        #     for j in range(i + 1, self.num_uavs):
        #         distance = np.linalg.norm(np.array(final_positions_real[i]) - np.array(final_positions_real[j]))
        #         if distance < 20:
        #             return 0  # Retornar 0 se houver colisão

        # Retornar fitness médio
        # Retornar fitness médio
        avg_fit = total_fitness / self.num_timeslots if self.num_timeslots > 0 else 0

        # ✅ Nova forma de reward: compressão logarítmica suave
        reward = np.log10(1.0 + avg_fit)

        return float(reward)

    def _calculate_communication_capacity(self, antenna_angles, alignments, positions, jammer_position):
        """Mesma função do teu código"""
        communication_capacity = np.zeros((self.num_uavs, self.num_uavs))
        interference_matrix = np.full((self.num_uavs, self.num_uavs), self.P_INTERFERENCE_DBM)

        for i in range(self.num_uavs):
            null_dir_i = antenna_angles[i]

            dx_jam = jammer_position[0] - positions[i][0]
            dy_jam = jammer_position[1] - positions[i][1]
            dir_jammer_to_uav = np.degrees(np.arctan2(dy_jam, dx_jam)) % 360

            G_jammer_i = realistic_antenna_gain(dir_jammer_to_uav, null_dir_i)

            dist_jammer_i = np.linalg.norm(positions[i] - jammer_position)
            P_interf_dBm_i = interference_from_jammer(self.P_INTERFERENCE_DBM, G_jammer_i, 
                                                    self.lambda_0, dist_jammer_i)

            interference_matrix[i, :] = P_interf_dBm_i

        for i in range(self.num_uavs):
            for j in range(self.num_uavs):
                if i != j:
                    null_dir_i = antenna_angles[i]
                    null_dir_j = antenna_angles[j]

                    dx_ij = positions[j][0] - positions[i][0]
                    dy_ij = positions[j][1] - positions[i][1]
                    dir_tx_to_rx = np.degrees(np.arctan2(dy_ij, dx_ij)) % 360

                    dx_ji = positions[i][0] - positions[j][0]
                    dy_ji = positions[i][1] - positions[j][1]
                    dir_rx_to_tx = np.degrees(np.arctan2(dy_ji, dx_ji)) % 360

                    G_tx = realistic_antenna_gain(dir_tx_to_rx, null_dir_i)
                    G_rx = realistic_antenna_gain(dir_rx_to_tx, null_dir_j)

                    capacity = calcular_capacidade_link(
                        pos1=positions[i],
                        pos2=positions[j],
                        lambda_0=self.lambda_0,
                        P_tx_dBm=self.TRANSMIT_POWER_DBM,
                        G_tx_dB=G_tx,
                        G_rx_dB=G_rx,
                        P_interference_dBm=interference_matrix[i, j],
                        P_noise_dBm=self.P_NOISE_DBM,
                        bandwidth=self.BANDWIDTH
                    )

                    communication_capacity[i][j] = capacity

        return communication_capacity, interference_matrix
    
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)

        # 🔹 Gera posições aleatórias para os UAVs dentro da área inicial
        self.uav_positions_real = np.random.uniform(
            low=[0, 0],
            high=[60, 60],
            size=(self.num_uavs, 2)
        )

        # 🔹 Gera posição aleatória do jammer (x entre 45 e 75)
        jammer_x = np.random.uniform(0, 120)
        self.jammer_position_real = np.array([jammer_x, 500.0])

        # 🔹 Normaliza tudo para observação inicial
        uav_positions_norm = []
        for pos in self.uav_positions_real:
            norm_pos = self._normalize_uav_position(pos, 'initial')
            uav_positions_norm.extend(norm_pos)

        jammer_norm = self._normalize_jammer_position(self.jammer_position_real)
        obs = np.concatenate([uav_positions_norm, jammer_norm])

        return obs.astype(np.float32), {}


    def step(self, action):
        # Desnormalizar ações (para coordenadas reais)
        action_reshaped = action.reshape(self.num_uavs, 2)
        final_positions_real = np.array([
            self._denormalize_uav_position(a, 'target') for a in action_reshaped
        ])

        # Guardar posições iniciais
        initial_positions_copy = self.uav_positions_real.copy()

        # Calcular reward com base nas posições finais
        reward = self._calculate_reward(final_positions_real)

        # Atualizar estado interno
        self.uav_positions_real = final_positions_real

        # Nova observação = posições finais + jammer
        uav_positions_norm = []
        for pos in final_positions_real:
            uav_positions_norm.extend(self._normalize_uav_position(pos, 'target'))
        jammer_norm = self._normalize_jammer_position(self.jammer_position_real)
        obs = np.concatenate([uav_positions_norm, jammer_norm])

        # Episódio termina após uma ação
        terminated = True
        truncated = False

        info = {
            'initial_positions_real': initial_positions_copy,
            'final_positions_real': final_positions_real,
            'jammer_position_real': self.jammer_position_real
        }

        return obs.astype(np.float32), reward, terminated, truncated, info



# Função para salvar (adiciona antes do if __name__ == "__main__":)
def save_best_solution_to_file(initial_positions, final_positions, num_timeslots=6, filename="best_solution_rl.txt"):
    """
    Salva a melhor solução do RL no mesmo formato do GA
    """
    # Ler o conteúdo existente do arquivo
    try:
        with open(filename, "r") as file:
            existing_content = file.readlines()
            print("Conteúdo existente no arquivo:")
            for line in existing_content:
                print(line.strip())
    except FileNotFoundError:
        print("O arquivo não existe. Criando um novo arquivo.")

    # Criar a solução completa (mesmo formato do GA)
    best_solution = []
    
    # Para cada timeslot, calcular posições interpoladas
    for t in range(num_timeslots):
        for uav in range(len(initial_positions)):
            start_pos = initial_positions[uav]
            end_pos = final_positions[uav]
            
            # Interpolação linear (mesma lógica do GA)
            alpha = t / (num_timeslots - 1) if num_timeslots > 1 else 0
            x = start_pos[0] + alpha * (end_pos[0] - start_pos[0])
            y = start_pos[1] + alpha * (end_pos[1] - start_pos[1])
            
            best_solution.extend([x, y])

    # Escrever a nova melhor solução no final do arquivo
    with open(filename, "a") as file:
        individual_str = ' '.join([f"{best_solution[i]:.2f}" for i in range(len(best_solution))])
        file.write(f"{individual_str}\n\n")
    
    print(f"Solução salva com {len(best_solution)} valores")
    return best_solution

# Exemplo de uso
if __name__ == "__main__":
    env = UAVCommunicationEnv()
    
    # Resetar o ambiente (usa posições fixas internas)
    obs, _ = env.reset()
    
    print(f"Observação inicial: {obs}")
    print(f"Shape observação: {obs.shape}")
    
    # Ação aleatória
    action = env.action_space.sample()
    print(f"Ação: {action}")
    
    # Executar a ação
    obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"\nReward: {reward:.4f}")
    print("\nPosições iniciais reais:")
    print(np.round(info['initial_positions_real'], 2))
    
    print("\nPosições finais reais:")
    print(np.round(info['final_positions_real'], 2))
    
    print(f"\nPosição jammer: {info['jammer_position_real']}")


Observação inicial: [0.64701945 0.380874   0.6077441  0.99251425 0.70377827 0.47358274
 0.6233031  0.08293363 0.87315226 1.        ]
Shape observação: (10,)
Ação: [0.8620648  0.21697432 0.7601991  0.4199488  0.96778554 0.31921393
 0.7318006  0.94527453]

Reward: 5.7352

Posições iniciais reais:
[[38.82 22.85]
 [36.46 59.55]
 [42.23 28.41]
 [37.4   4.98]]

Posições finais reais:
[[111.72  13.02]
 [105.61  25.2 ]
 [118.07  19.15]
 [103.91  56.72]]

Posição jammer: [104.77826996 500.        ]


In [104]:
env = UAVCommunicationEnv()
for _ in range(3):
    obs, _ = env.reset()
    print("Jam:", env.jammer_position_real)
    print("UAVs:", np.round(env.uav_positions_real, 2))
    print("-" * 50)


Jam: [110.96159941 500.        ]
UAVs: [[57.07 47.13]
 [47.22 29.08]
 [ 8.11 58.78]
 [52.64 56.48]]
--------------------------------------------------
Jam: [ 46.63726537 500.        ]
UAVs: [[38.48 49.42]
 [40.88 46.42]
 [17.62 12.65]
 [53.09 50.51]]
--------------------------------------------------
Jam: [ 63.52267894 500.        ]
UAVs: [[33.17 57.22]
 [44.32 45.24]
 [47.78  9.08]
 [58.64 49.72]]
--------------------------------------------------


In [105]:
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env

env = make_vec_env(UAVCommunicationEnv, n_envs=4)
model = PPO.load("policy_finetuned.zip", env=env)

model.learn(total_timesteps=300_000)
model.save("policy_generalized.zip")

print("✅ Treino com randomização concluído!")


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 1        |
|    ep_rew_mean     | 15.4     |
| time/              |          |
|    fps             | 139      |
|    iterations      | 1        |
|    time_elapsed    | 58       |
|    total_timesteps | 8192     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 1           |
|    ep_rew_mean          | 15.4        |
| time/                   |             |
|    fps                  | 121         |
|    iterations           | 2           |
|    time_elapsed         | 134         |
|    total_timesteps      | 16384       |
| train/                  |             |
|    approx_kl            | 0.011864318 |
|    clip_fraction        | 0.204       |
|    clip_range           | 0.2         |
|    entropy_loss         | -2.04       |
|    explained_variance   | -0.0108     |
|    learning_rate        | 0.

In [106]:
from stable_baselines3 import PPO
import numpy as np

model = PPO.load("policy_generalized.zip")
env = UAVCommunicationEnv()

for i in range(5):
    obs, _ = env.reset()
    action, _ = model.predict(obs, deterministic=True)
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Teste {i+1}")
    print(f"Reward: {reward:.3f}")
    print("Jammer:", env.jammer_position_real)
    print("UAVs iniciais:", np.round(info['initial_positions_real'], 2))
    print("UAVs finais:", np.round(info['final_positions_real'], 2))
    print("-" * 70)


Teste 1
Reward: 15.457
Jammer: [ 48.77697862 500.        ]
UAVs iniciais: [[57.43 27.82]
 [31.44 16.97]
 [27.3  40.78]
 [29.33 32.7 ]]
UAVs finais: [[60.  0.]
 [60.  0.]
 [60.  0.]
 [60.  0.]]
----------------------------------------------------------------------
Teste 2
Reward: 15.420
Jammer: [ 66.38634279 500.        ]
UAVs iniciais: [[35.73 46.66]
 [40.49 36.52]
 [57.78  1.74]
 [46.67 19.6 ]]
UAVs finais: [[60.  0.]
 [60.  0.]
 [60.  0.]
 [60.  0.]]
----------------------------------------------------------------------
Teste 3
Reward: 15.518
Jammer: [ 15.86727884 500.        ]
UAVs iniciais: [[33.53 28.5 ]
 [34.18 13.44]
 [30.    9.22]
 [35.73 54.6 ]]
UAVs finais: [[60.  0.]
 [60.  0.]
 [60.  0.]
 [60.  0.]]
----------------------------------------------------------------------
Teste 4
Reward: 15.324
Jammer: [104.61797067 500.        ]
UAVs iniciais: [[26.23 27.16]
 [14.47 28.68]
 [18.29 14.06]
 [31.23 14.16]]
UAVs finais: [[60.  0.]
 [60.  0.]
 [60.  0.]
 [60.  0.]]
---------------

In [109]:
from stable_baselines3 import PPO
import numpy as np

model = PPO.load("policy_generalized.zip")
env = UAVCommunicationEnv()

# Define as tuas posições iniciais (exemplo)
custom_positions = np.array([
    [0, 0],
    [60, 60],
    [60, 40],
    [40, 60]
])

# Define a posição do jammer (opcional)
custom_jammer = np.array([70.0, 500.0])

# Reset do ambiente
obs, _ = env.reset()

# Sobrescreve as posições
env.uav_positions_real = custom_positions
env.jammer_position_real = custom_jammer

# Recalcula a observação normalizada
uav_positions_norm = []
for pos in env.uav_positions_real:
    uav_positions_norm.extend(env._normalize_uav_position(pos, 'initial'))
jammer_norm = env._normalize_jammer_position(env.jammer_position_real)
obs = np.concatenate([uav_positions_norm, jammer_norm]).astype(np.float32)

# Testa com a política
action, _ = model.predict(obs, deterministic=True)
obs, reward, terminated, truncated, info = env.step(action)

print(f"Reward: {reward:.3f}")
print("Jammer:", env.jammer_position_real)
print("UAVs iniciais:", np.round(info['initial_positions_real'], 2))
print("UAVs finais:", np.round(info['final_positions_real'], 2))


Reward: 15.412
Jammer: [ 70. 500.]
UAVs iniciais: [[ 0  0]
 [60 60]
 [60 40]
 [40 60]]
UAVs finais: [[60.  0.]
 [60.  0.]
 [60.  0.]
 [60.  0.]]


# analisar

In [110]:
#escrever dataset com rl 1

In [113]:
import pandas as pd
import numpy as np
from stable_baselines3 import PPO

# === 1. Carregar modelo e ambiente ===
model_path = "policy_generalized.zip"
model = PPO.load(model_path)
env = UAVCommunicationEnv()

# === 2. Carregar dataset original ===
csv_path = "dataset_combined_com_permutacoes.csv"  # <-- ajusta o nome conforme o teu
df = pd.read_csv(csv_path)

# === 3. Preparar lista para armazenar as novas linhas ===
new_rows = []

# === 4. Loop por cada linha do CSV ===
for _, row in df.iterrows():
    # ----- Extrair jammer -----
    jammer_real = np.array([row["Jammer_X"], row["Jammer_Y"]])

    # ----- Extrair posições iniciais -----
    uav_positions_real = []
    for i in range(1, 5):
        uav_positions_real.append(np.array([row[f"Initial_X{i}"], row[f"Initial_Y{i}"]]))
    uav_positions_real = np.array(uav_positions_real)

    # ----- Normalizar observação -----
    uav_positions_norm = []
    for pos in uav_positions_real:
        uav_positions_norm.extend(env._normalize_uav_position(pos, "initial"))
    jammer_norm = env._normalize_jammer_position(jammer_real)

    # ✅ Cria vetor de observação com shape (10,) (sem collision_level)
    obs = np.concatenate([uav_positions_norm, jammer_norm]).astype(np.float32)

    # ----- Predição da política -----
    action, _ = model.predict(obs, deterministic=True)

    # ----- Desnormalizar ação para coordenadas reais -----
    action_reshaped = action.reshape(4, 2)
    predicted_final_positions = [
        env._denormalize_uav_position(a, "target") for a in action_reshaped
    ]
    predicted_final_positions = np.array(predicted_final_positions)

    # ----- Extrair posições finais reais -----
    real_final_positions = []
    for i in range(1, 5):
        real_final_positions.append(np.array([row[f"Final_X{i}"], row[f"Final_Y{i}"]]))
    real_final_positions = np.array(real_final_positions)

    # ----- Montar nova linha -----
    new_row = {
        "Jammer_X": row["Jammer_X"],
        "Jammer_Y": row["Jammer_Y"],
    }

    # Initials
    for i in range(4):
        new_row[f"Initial_X{i+1}"] = uav_positions_real[i][0]
        new_row[f"Initial_Y{i+1}"] = uav_positions_real[i][1]

    # Predicted Finals
    for i in range(4):
        new_row[f"Predicted_Final_X{i+1}"] = predicted_final_positions[i][0]
        new_row[f"Predicted_Final_Y{i+1}"] = predicted_final_positions[i][1]

    # Real Finals
    for i in range(4):
        new_row[f"Real_Final_X{i+1}"] = real_final_positions[i][0]
        new_row[f"Real_Final_Y{i+1}"] = real_final_positions[i][1]

    new_rows.append(new_row)

# === 5. Criar DataFrame final ===
new_df = pd.DataFrame(new_rows)

# === 6. Guardar novo CSV ===
output_path = "uav_predictions_generalized.csv"
new_df.to_csv(output_path, index=False)
print(f"✅ Novo CSV gerado com sucesso: {output_path}")
print(f"Total de linhas: {len(new_df)}")


✅ Novo CSV gerado com sucesso: uav_predictions_generalized.csv
Total de linhas: 2400


In [114]:
import numpy as np
import networkx as nx
import pandas as pd

def has_collision_final_positions(final_positions, min_distance=20):
    """
    nao verifica
    """
    
    return False  # Sem colisões

def analisar_comunicacoes_interpolado_sem_verificacao_colisoes(initial_positions, final_positions, num_uavs, num_timeslots, jammer_position):
    """
    Analisa comunicações interpolando entre posições inicial e final
    SEM VERIFICAR COLISÕES (para Análise 1)
    """
    
    def criar_best_solution_interpolado(initial_pos, final_pos, num_uavs, num_timeslots):
        """
        Cria best_solution interpolando linearmente entre posições inicial e final
        """
        best_solution = []
        
        for t in range(num_timeslots):
            # Calcular fator de interpolação (0 no início, 1 no final)
            if num_timeslots == 1:
                alpha = 1.0  # Se só há 1 timeslot, usar posição final
            else:
                alpha = t / (num_timeslots - 1)
            
            # Para cada UAV
            for uav in range(num_uavs):
                start_x, start_y = initial_pos[uav]
                end_x, end_y = final_pos[uav]
                
                # Interpolação linear
                x = start_x + alpha * (end_x - start_x)
                y = start_y + alpha * (end_y - start_y)
                
                # Adicionar ao best_solution
                best_solution.extend([x, y])
        
        return best_solution
    
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # CRIAR BEST_SOLUTION ATRAVÉS DE INTERPOLAÇÃO
    best_solution = criar_best_solution_interpolado(initial_positions, final_positions, num_uavs, num_timeslots)
    
    # SEM VERIFICAÇÃO DE COLISÕES - continuar sempre com a análise
    resultados = []
    todos_c_media = []
    todos_c_min = []
    
    # Analisar cada timeslot
    for t in range(num_timeslots):
        positions = []
        angles = []
        
        for i in range(num_uavs):
            idx_pos = t * num_uavs * 2 + i * 2
            x, y = best_solution[idx_pos:idx_pos+2]
            positions.append(np.array([x, y]))
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            angles.append(angle)
        
        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
        
        # Gerar métricas detalhadas
        metricas = avaliar_grafo(comm_matrix)
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Adicionar às listas do epoch
        todos_c_media.append(C_media_total)
        todos_c_min.append(C_min_total)
        
        resultados.append({
            'epoch': 1,
            'timeslot': t+1,
            'metricas': metricas,
            'C_media_total': C_media_total,
            'C_min_total': C_min_total
        })
    
    # Calcular fitness final
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []
    
    for i in range(len(todos_c_media)):
        C_media = todos_c_media[i]
        C_min = todos_c_min[i]
        fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
        fitness_por_timeslot.append(fitness_timeslot)
    
    # Fitness médio do epoch
    fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
    
    # Calcular também as médias para informação
    media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
    media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch

def analisar_comunicacoes_interpolado_final(initial_positions, final_positions, num_uavs, num_timeslots, jammer_position):
    """
    Analisa comunicações interpolando entre posições inicial e final
    VERIFICA COLISÕES APENAS NAS POSIÇÕES FINAIS
    """
    
    def criar_best_solution_interpolado(initial_pos, final_pos, num_uavs, num_timeslots):
        """
        Cria best_solution interpolando linearmente entre posições inicial e final
        """
        best_solution = []
        
        for t in range(num_timeslots):
            # Calcular fator de interpolação (0 no início, 1 no final)
            if num_timeslots == 1:
                alpha = 1.0  # Se só há 1 timeslot, usar posição final
            else:
                alpha = t / (num_timeslots - 1)
            
            # Para cada UAV
            for uav in range(num_uavs):
                start_x, start_y = initial_pos[uav]
                end_x, end_y = final_pos[uav]
                
                # Interpolação linear
                x = start_x + alpha * (end_x - start_x)
                y = start_y + alpha * (end_y - start_y)
                
                # Adicionar ao best_solution
                best_solution.extend([x, y])
        
        return best_solution
    
    def avaliar_grafo(comm_matrix):
        G = nx.DiGraph()
        
        # 1. Adicionar arestas bidirecionais com capacidades
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j and comm_matrix[i][j] > 0:
                    G.add_edge(i, j, capacity=comm_matrix[i][j])
        
        # 2. Cálculos de métricas
        metricas = {
            'capacidades': [],
            'caminhos_minimos': {},
            'bottlenecks': {},
            'conectividade': None,
            'links_usados': set(),
            'links_nao_usados': set()
        }
        
        # Preencher métricas
        for i in range(num_uavs):
            for j in range(num_uavs):
                if i != j:
                    # Calcular caminhos mínimos (1/capacity como peso)
                    try:
                        path = nx.shortest_path(G, source=i, target=j, weight=lambda u, v, d: 1/d['capacity'])
                        capacidade_min = min(G[u][v]['capacity'] for u, v in zip(path[:-1], path[1:]))
                        
                        metricas['caminhos_minimos'][(i,j)] = {
                            'path': path,
                            'capacidade': capacidade_min
                        }
                        
                        # Adicionar links usados
                        for u, v in zip(path[:-1], path[1:]):
                            metricas['links_usados'].add((u, v))
                            metricas['links_usados'].add((v, u))  # Adicionar a aresta reversa
                    except:
                        pass
        
        # Calcular bottlenecks para cada nó
        for node in G.nodes():
            metricas['bottlenecks'][node] = min(
                [d['capacity'] for _, _, d in G.edges(node, data=True)],
                default=0
            )
        
        # Verificar conectividade
        metricas['conectividade'] = nx.is_strongly_connected(G)
        
        # Coletar todas as capacidades
        metricas['capacidades'] = [d['capacity'] for _, _, d in G.edges(data=True)]
        
        # Identificar links não usados
        all_links = {(i, j) for i in range(num_uavs) for j in range(num_uavs) if i != j}
        metricas['links_nao_usados'] = all_links - metricas['links_usados']
        
        return metricas

    # CRIAR BEST_SOLUTION ATRAVÉS DE INTERPOLAÇÃO
    best_solution = criar_best_solution_interpolado(initial_positions, final_positions, num_uavs, num_timeslots)
    
    # 🆕 VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS (não ao longo do caminho)
    tem_colisoes = has_collision_final_positions(final_positions)
    
    if tem_colisoes:
        return [(0.0, "Solução com colisões nas posições finais")], 0.0, 0.0, 0.0
    
    # Se não há colisões nas posições finais, continuar com a análise normal
    resultados = []
    todos_c_media = []
    todos_c_min = []
    
    # Analisar cada timeslot
    for t in range(num_timeslots):
        positions = []
        angles = []
        
        for i in range(num_uavs):
            idx_pos = t * num_uavs * 2 + i * 2
            x, y = best_solution[idx_pos:idx_pos+2]
            positions.append(np.array([x, y]))
            
            # Calcular ângulo dinamicamente
            angle = np.degrees(np.arctan2(jammer_position[1] - y, jammer_position[0] - x))
            angles.append(angle)
        
        # Calcular matriz de comunicação
        comm_matrix, _ = calculate_communication_capacity(angles, [1.0]*num_uavs, positions, jammer_position)
        
        # Gerar métricas detalhadas
        metricas = avaliar_grafo(comm_matrix)
        
        # Cálculo do fitness usando apenas links usados
        capacidades_usadas = [comm_matrix[u][v] for u, v in metricas['links_usados'] if u < num_uavs and v < num_uavs]
        if capacidades_usadas:
            C_media_total = np.mean(capacidades_usadas)
            C_min_total = min(capacidades_usadas)
        else:
            C_media_total = 0
            C_min_total = 0
        
        # Adicionar às listas do epoch
        todos_c_media.append(C_media_total)
        todos_c_min.append(C_min_total)
        
        resultados.append({
            'epoch': 1,
            'timeslot': t+1,
            'metricas': metricas,
            'C_media_total': C_media_total,
            'C_min_total': C_min_total
        })
    
    # Calcular fitness final
    alpha, beta = 1.0, 1.0
    fitness_por_timeslot = []
    
    for i in range(len(todos_c_media)):
        C_media = todos_c_media[i]
        C_min = todos_c_min[i]
        fitness_timeslot = (C_media ** alpha) * (C_min ** beta)
        fitness_por_timeslot.append(fitness_timeslot)
    
    # Fitness médio do epoch
    fitness_epoch = np.mean(fitness_por_timeslot) if fitness_por_timeslot else 0
    
        # Calcular também as médias para informação
    media_c_media_epoch = np.mean(todos_c_media) if todos_c_media else 0
    media_c_min_epoch = np.mean(todos_c_min) if todos_c_min else 0
    
    return resultados, fitness_epoch, media_c_media_epoch, media_c_min_epoch


In [118]:
def carregar_e_analisar_dataset_final(caminho_dataset, num_uavs=4, num_timeslots=6):
    """
    Carrega o dataset e analisa comunicações para valores reais e preditos
    TRÊS ANÁLISES com verificação de colisões apenas nas posições finais:
    1. Todas as linhas (valores reais)
    2. Apenas sem colisões
    3. Todas as linhas com colisões = 0
    """
    
    # CARREGAR O DATASET
    df = pd.read_csv(caminho_dataset)
    
    # ANÁLISE 1: Todas as linhas (valores reais)
    fitness_reais = []
    fitness_preditos = []
    c_media_reais = []
    c_media_preditos = []
    c_min_reais = []
    c_min_preditos = []
    
    # ANÁLISE 2: Apenas sem colisões
    fitness_reais_sem_colisoes = []
    fitness_preditos_sem_colisoes = []
    c_media_reais_sem_colisoes = []
    c_media_preditos_sem_colisoes = []
    c_min_reais_sem_colisoes = []
    c_min_preditos_sem_colisoes = []
    
    # ANÁLISE 3: Todas as linhas, mas colisões = 0
    fitness_reais_colisoes_zero = []
    fitness_preditos_colisoes_zero = []
    c_media_reais_colisoes_zero = []
    c_media_preditos_colisoes_zero = []
    c_min_reais_colisoes_zero = []
    c_min_preditos_colisoes_zero = []
    
    linhas_sem_colisoes = []
    total_linhas_sem_colisoes = 0
    
    # ANALISAR CADA LINHA DO DATASET
    for index, row in df.iterrows():
        # Extrair posições
        initial_positions = [
            (row['Initial_X1'], row['Initial_Y1']),
            (row['Initial_X2'], row['Initial_Y2']),
            (row['Initial_X3'], row['Initial_Y3']),
            (row['Initial_X4'], row['Initial_Y4'])
        ]
        
        final_positions_preditas = [
            (row['Predicted_Final_X1'], row['Predicted_Final_Y1']),
            (row['Predicted_Final_X2'], row['Predicted_Final_Y2']),
            (row['Predicted_Final_X3'], row['Predicted_Final_Y3']),
            (row['Predicted_Final_X4'], row['Predicted_Final_Y4'])
        ]
        
        final_positions_reais = [
            (row['Real_Final_X1'], row['Real_Final_Y1']),
            (row['Real_Final_X2'], row['Real_Final_Y2']),
            (row['Real_Final_X3'], row['Real_Final_Y3']),
            (row['Real_Final_X4'], row['Real_Final_Y4'])
        ]
        
        jammer_position = [row['Jammer_X'], row['Jammer_Y']]
        
        # VERIFICAR COLISÕES APENAS NAS POSIÇÕES FINAIS
        tem_colisoes_predito = has_collision_final_positions(final_positions_preditas)
        tem_colisoes_real = has_collision_final_positions(final_positions_reais)
        
        # CALCULAR VALORES REAIS DE COMUNICAÇÃO (mesmo com colisões) - USANDO FUNÇÃO SEM VERIFICAÇÃO
        resultados_pred, fitness_pred, c_media_pred, c_min_pred = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_preditas,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        resultados_real, fitness_real, c_media_real, c_min_real = analisar_comunicacoes_interpolado_sem_verificacao_colisoes(
            initial_positions=initial_positions,
            final_positions=final_positions_reais,
            num_uavs=num_uavs,
            num_timeslots=num_timeslots,
            jammer_position=jammer_position
        )
        
        # ANÁLISE 1: Armazenar valores reais (independente de colisões)
        fitness_preditos.append(fitness_pred)
        fitness_reais.append(fitness_real)
        c_media_preditos.append(c_media_pred)
        c_media_reais.append(c_media_real)
        c_min_preditos.append(c_min_pred)
        c_min_reais.append(c_min_real)
        
        # ANÁLISE 2: Se não há colisões, adicionar às listas especiais
        if not tem_colisoes_predito:
            fitness_preditos_sem_colisoes.append(fitness_pred)
            fitness_reais_sem_colisoes.append(fitness_real)
            c_media_preditos_sem_colisoes.append(c_media_pred)
            c_media_reais_sem_colisoes.append(c_media_real)
            c_min_preditos_sem_colisoes.append(c_min_pred)
            c_min_reais_sem_colisoes.append(c_min_real)
            
            linhas_sem_colisoes.append(index + 1)
            total_linhas_sem_colisoes += 1
        
        # ANÁLISE 3: Colisões = 0
        # Para preditos
        if tem_colisoes_predito:
            fitness_preditos_colisoes_zero.append(0.0)
            c_media_preditos_colisoes_zero.append(0.0)
            c_min_preditos_colisoes_zero.append(0.0)
        else:
            fitness_preditos_colisoes_zero.append(fitness_pred)
            c_media_preditos_colisoes_zero.append(c_media_pred)
            c_min_preditos_colisoes_zero.append(c_min_pred)
        
        # Para reais
        if tem_colisoes_real:
            fitness_reais_colisoes_zero.append(0.0)
            c_media_reais_colisoes_zero.append(0.0)
            c_min_reais_colisoes_zero.append(0.0)
        else:
            fitness_reais_colisoes_zero.append(fitness_real)
            c_media_reais_colisoes_zero.append(c_media_real)
            c_min_reais_colisoes_zero.append(c_min_real)
    
    # CALCULAR MÉDIAS
    media_fitness_real_1 = np.mean(fitness_reais)
    media_fitness_predito_1 = np.mean(fitness_preditos)
    media_c_media_real_1 = np.mean(c_media_reais)
    media_c_media_predito_1 = np.mean(c_media_preditos)
    media_c_min_real_1 = np.mean(c_min_reais)
    media_c_min_predito_1 = np.mean(c_min_preditos)
    
    media_fitness_real_3 = np.mean(fitness_reais_colisoes_zero)
    media_fitness_predito_3 = np.mean(fitness_preditos_colisoes_zero)
    media_c_media_real_3 = np.mean(c_media_reais_colisoes_zero)
    media_c_media_predito_3 = np.mean(c_media_preditos_colisoes_zero)
    media_c_min_real_3 = np.mean(c_min_reais_colisoes_zero)
    media_c_min_predito_3 = np.mean(c_min_preditos_colisoes_zero)
    
    # ANÁLISE 1: RESULTADOS PARA TODAS AS LINHAS
    print(f"============================================================")
    print(f"ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)")
    print(f"============================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_1:.4f} | Predito={media_fitness_predito_1:.4f} | Diff={abs(media_fitness_real_1 - media_fitness_predito_1):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_1:.2f} | Predito={media_c_media_predito_1:.2f} | Diff={abs(media_c_media_real_1 - media_c_media_predito_1):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_1:.2f} | Predito={media_c_min_predito_1:.2f} | Diff={abs(media_c_min_real_1 - media_c_min_predito_1):.2f}")
    
    # ANÁLISE 2: APENAS SEM COLISÕES
    if total_linhas_sem_colisoes > 0:
        media_fitness_real_2 = np.mean(fitness_reais_sem_colisoes)
        media_fitness_predito_2 = np.mean(fitness_preditos_sem_colisoes)
        media_c_media_real_2 = np.mean(c_media_reais_sem_colisoes)
        media_c_media_predito_2 = np.mean(c_media_preditos_sem_colisoes)
        media_c_min_real_2 = np.mean(c_min_reais_sem_colisoes)
        media_c_min_predito_2 = np.mean(c_min_preditos_sem_colisoes)
        
        print(f"\n======================================================================")
        print(f"ANÁLISE 2 - APENAS LINHAS SEM COLISÕES ({total_linhas_sem_colisoes}/{len(df)} linhas)")
        print(f"======================================================================")
        print(f"🎯 FITNESS: Real={media_fitness_real_2:.4f} | Predito={media_fitness_predito_2:.4f} | Diff={abs(media_fitness_real_2 - media_fitness_predito_2):.4f}")
        print(f"📊 C_MÉDIA: Real={media_c_media_real_2:.2f} | Predito={media_c_media_predito_2:.2f} | Diff={abs(media_c_media_real_2 - media_c_media_predito_2):.2f}")
        print(f"📉 C_MIN: Real={media_c_min_real_2:.2f} | Predito={media_c_min_predito_2:.2f} | Diff={abs(media_c_min_real_2 - media_c_min_predito_2):.2f}")
    
    # ANÁLISE 3: COLISÕES = 0
    print(f"\n======================================================================")
    print(f"ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)")
    print(f"======================================================================")
    print(f"🎯 FITNESS: Real={media_fitness_real_3:.4f} | Predito={media_fitness_predito_3:.4f} | Diff={abs(media_fitness_real_3 - media_fitness_predito_3):.4f}")
    print(f"📊 C_MÉDIA: Real={media_c_media_real_3:.2f} | Predito={media_c_media_predito_3:.2f} | Diff={abs(media_c_media_real_3 - media_c_media_predito_3):.2f}")
    print(f"📉 C_MIN: Real={media_c_min_real_3:.2f} | Predito={media_c_min_predito_3:.2f} | Diff={abs(media_c_min_real_3 - media_c_min_predito_3):.2f}")

# USO
if __name__ == "__main__":
    caminho_do_dataset = "uav_predictions_generalized.csv"
    carregar_e_analisar_dataset_final(caminho_dataset=caminho_do_dataset, num_uavs=4, num_timeslots=6)


ANÁLISE 1 - TODAS AS LINHAS (valores reais de comunicação)
🎯 FITNESS: Real=328578688.4350 | Predito=2707716003063408.5000 | Diff=2707715674484720.0000
📊 C_MÉDIA: Real=18089.93 | Predito=21155387.48 | Diff=21137297.56
📉 C_MIN: Real=7415.45 | Predito=21150290.90 | Diff=21142875.45

ANÁLISE 2 - APENAS LINHAS SEM COLISÕES (2400/2400 linhas)
🎯 FITNESS: Real=328578688.4350 | Predito=2707716003063408.5000 | Diff=2707715674484720.0000
📊 C_MÉDIA: Real=18089.93 | Predito=21155387.48 | Diff=21137297.56
📉 C_MIN: Real=7415.45 | Predito=21150290.90 | Diff=21142875.45

ANÁLISE 3 - TODAS AS LINHAS (colisões = 0)
🎯 FITNESS: Real=328578688.4350 | Predito=2707716003063408.5000 | Diff=2707715674484720.0000
📊 C_MÉDIA: Real=18089.93 | Predito=21155387.48 | Diff=21137297.56
📉 C_MIN: Real=7415.45 | Predito=21150290.90 | Diff=21142875.45


In [1]:
import pandas as pd
import numpy as np
import os
import glob

def calculate_mean_euclidean_distance_single(dataset_file):
    """
    Calcula a Mean Euclidean Distance para um único ficheiro
    
    Args:
        dataset_file: Caminho do ficheiro CSV
    
    Returns:
        dict: Estatísticas da distância euclidiana
    """
    
    try:
        # Carregar dataset
        df = pd.read_csv(dataset_file)
        
        # Lista para armazenar todas as distâncias
        all_distances = []
        
        # Calcular distância euclidiana para cada linha e cada UAV
        for index, row in df.iterrows():
            for uav in range(1, 5):  # UAVs 1, 2, 3, 4
                # Posições preditas
                pred_x = row[f'Predicted_Final_X{uav}']
                pred_y = row[f'Predicted_Final_Y{uav}']
                
                # Posições reais
                real_x = row[f'Real_Final_X{uav}']
                real_y = row[f'Real_Final_Y{uav}']
                
                # Calcular distância euclidiana
                distance = np.sqrt((pred_x - real_x)**2 + (pred_y - real_y)**2)
                all_distances.append(distance)
        
        # Calcular estatísticas
        results = {
            'filename': os.path.basename(dataset_file),
            'total_predictions': len(all_distances),
            'total_lines': len(df),
            'mean_distance': np.mean(all_distances),
            'std_distance': np.std(all_distances),
            'min_distance': np.min(all_distances),
            'max_distance': np.max(all_distances),
            'median_distance': np.median(all_distances)
        }
        
        return results
        
    except Exception as e:
        print(f"❌ Erro ao processar {dataset_file}: {e}")
        return None

def analyze_multiple_files(file_list=None, directory=None, pattern="*.csv"):
    """
    Analisa múltiplos ficheiros e calcula Mean Euclidean Distance
    
    Args:
        file_list: Lista de caminhos de ficheiros (opcional)
        directory: Diretório para procurar ficheiros (opcional)
        pattern: Padrão de ficheiros a procurar (default: "*.csv")
    """
    
    # Determinar lista de ficheiros
    if file_list:
        files = file_list
    elif directory:
        files = glob.glob(os.path.join(directory, pattern))
    else:
        print("❌ Deve fornecer file_list ou directory")
        return
    
    if not files:
        print("❌ Nenhum ficheiro encontrado")
        return
    
    print(f"🔍 Encontrados {len(files)} ficheiros para analisar")
    print("="*80)
    
    # Analisar cada ficheiro
    results = []
    
    for file_path in files:
        print(f"📊 Analisando: {os.path.basename(file_path)}")
        result = calculate_mean_euclidean_distance_single(file_path)
        
        if result:
            results.append(result)
            print(f"   ✅ Mean Euclidean Distance: {result['mean_distance']:.4f}")
        else:
            print(f"   ❌ Falhou")
        print()
    
    # Mostrar resumo comparativo
    if results:
        print("="*80)
        print("📈 RESUMO COMPARATIVO - MEAN EUCLIDEAN DISTANCE")
        print("="*80)
        
        # Cabeçalho
        print(f"{'Ficheiro':<40} {'Mean Dist':<12} {'Std':<10} {'Min':<10} {'Max':<10} {'Linhas':<8}")
        print("-" * 90)
        
        # Resultados
        for result in results:
            filename = result['filename'][:37] + "..." if len(result['filename']) > 40 else result['filename']
            print(f"{filename:<40} {result['mean_distance']:<12.4f} {result['std_distance']:<10.4f} "
                  f"{result['min_distance']:<10.4f} {result['max_distance']:<10.4f} {result['total_lines']:<8}")
        
        # Encontrar melhor e pior
        best = min(results, key=lambda x: x['mean_distance'])
        worst = max(results, key=lambda x: x['mean_distance'])
        
        print("\n" + "="*50)
        print("🏆 RANKING:")
        print(f"   🥇 MELHOR: {best['filename']} (Mean: {best['mean_distance']:.4f})")
        print(f"   🥉 PIOR:   {worst['filename']} (Mean: {worst['mean_distance']:.4f})")
        print(f"   📊 DIFERENÇA: {worst['mean_distance'] - best['mean_distance']:.4f}")
        
        return results
    
    else:
        print("❌ Nenhum ficheiro foi processado com sucesso")
        return None

# Função de conveniência para usar facilmente
def quick_analysis(files):
    """
    Análise rápida de uma lista de ficheiros
    
    Args:
        files: Lista de caminhos de ficheiros
    """
    return analyze_multiple_files(file_list=files)

# Executar
if __name__ == "__main__":
    
    # OPÇÃO 1: Lista específica de ficheiros
    files_to_analyze = [
        "uav_predictions_generalized.csv"
    ]
    
    print("🚀 ANÁLISE DE MEAN EUCLIDEAN DISTANCE")
    print("="*80)
    
    results = quick_analysis(files_to_analyze)
    
    # OPÇÃO 2: Todos os ficheiros de um diretório
    # results = analyze_multiple_files(directory="class_results", pattern="*.csv")
    
    # OPÇÃO 3: Ficheiro individual
    # result = calculate_mean_euclidean_distance_single("inter_results/1n_knn.csv")
    # print(f"Mean Distance: {result['mean_distance']:.4f}")


🚀 ANÁLISE DE MEAN EUCLIDEAN DISTANCE
🔍 Encontrados 1 ficheiros para analisar
📊 Analisando: uav_predictions_generalized.csv
   ✅ Mean Euclidean Distance: 44.0773

📈 RESUMO COMPARATIVO - MEAN EUCLIDEAN DISTANCE
Ficheiro                                 Mean Dist    Std        Min        Max        Linhas  
------------------------------------------------------------------------------------------
uav_predictions_generalized.csv          44.0773      14.6082    5.6421     75.4611    2400    

🏆 RANKING:
   🥇 MELHOR: uav_predictions_generalized.csv (Mean: 44.0773)
   🥉 PIOR:   uav_predictions_generalized.csv (Mean: 44.0773)
   📊 DIFERENÇA: 0.0000
